In [15]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

print("YOLO Loaded Successfully")

YOLO Loaded Successfully


In [3]:
import sys

!"{sys.executable}" -m pip install numpy==1.26.4 opencv-python==4.10.0.84

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import numpy as np
import cv2
import matplotlib
from ultralytics import YOLO

print("NumPy version:", np.__version__)
print("OpenCV version:", cv2.__version__)
print("Matplotlib version:", matplotlib.__version__)

model = YOLO("yolov8n.pt")

print("YOLO loaded successfully!")

NumPy version: 1.26.4
OpenCV version: 4.10.0
Matplotlib version: 3.9.2
YOLO loaded successfully!


In [4]:
import os
import cv2

video_path = "busy_traffic.mp4.mp4"

print("Current folder:", os.getcwd())
print("File exists:", os.path.exists(video_path))

cap = cv2.VideoCapture(video_path)
print("Video opened:", cap.isOpened())

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("FPS:", fps)
print("Frame Count:", frame_count)

cap.release()

Current folder: D:\traffic_project
File exists: True
Video opened: True
FPS: 23.976023976023978
Frame Count: 466


In [3]:
import os

print("Current folder:", os.getcwd())
print("All files:")
for f in os.listdir():
    print(f)

Current folder: D:\traffic_project
All files:
.ipynb_checkpoints
adaptive_traffic_control_updated.ipynb
busy_traffic.mp4.mp4
yolov8n.pt


In [5]:
import cv2

video_path = "busy_traffic.mp4.mp4"

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

duration = frame_count / fps

print("FPS:", fps)
print("Frame Count:", frame_count)
print("Duration:", round(duration, 2), "seconds")

cap.release()

FPS: 23.976023976023978
Frame Count: 466
Duration: 19.44 seconds


In [20]:
vehicle_classes = ["car", "bus", "truck", "motorcycle"]

vehicle_count = 0

for box in results[0].boxes:
    cls_id = int(box.cls[0])
    cls_name = model.names[cls_id]

    if cls_name in vehicle_classes:
        vehicle_count += 1

print("Vehicle Count:", vehicle_count)

Vehicle Count: 41


In [1]:
import sys
print(sys.executable)

D:\traffic_project\.venv\Scripts\python.exe


# Adaptive Traffic Signal Control Using Edge AI and Local LLM Agent

This notebook implements an Edge AI-based adaptive traffic signal control prototype.

The system reads a local traffic video, samples frames at fixed intervals, and uses YOLOv8n to detect and count vehicles locally. Based on the average vehicle count in each control cycle, the system calculates an adaptive green light duration. After the green light and yellow light time, the next detection cycle starts automatically.

To continue the pipeline from detection to decision-making, this notebook also integrates a local lightweight LLM agent through Ollama using `qwen2.5:0.5b`. The LLM does not receive raw video frames. It only receives structured metadata, such as sample times, vehicle counts, average vehicle count, traffic state, green light time, and inference latency.

The traffic state and core control decision are determined by local rule-based logic for stability. The local LLM agent is used to explain the decision, generate an action, and provide a privacy note.

## Main Pipeline

Local traffic video  
→ Frame sampling  
→ YOLOv8n vehicle detection  
→ Vehicle counting  
→ Average vehicle count calculation  
→ Adaptive green light control  
→ Local LLM agent explanation  
→ Decision and action output

## Edge AI Design

This project is considered Edge AI because video processing, vehicle detection, traffic control, and LLM-based reasoning are all performed locally. Raw video is not uploaded to the cloud. Only structured metadata is passed to the local LLM agent.

## Output Files

After running the main code, the notebook generates:

- `traffic_control_with_llm_results_fixed.csv`
- `yolo_detection_sample.jpg`

In [2]:
from ultralytics import YOLO
import cv2
import requests
import pandas as pd

model = YOLO("yolov8n.pt")
print("YOLO loaded successfully!")

YOLO loaded successfully!


In [4]:
# ============================================================
# Adaptive Traffic Signal Control with YOLOv8n + Local LLM Agent
# ============================================================
# Project goal:
# This program detects vehicles from a local traffic video, calculates
# adaptive green light duration, and uses a local lightweight LLM agent
# to explain the traffic situation and generate a decision.
#
# Edge AI design:
# - YOLOv8n runs locally for vehicle detection.
# - Traffic state and green light time are calculated locally.
# - The local LLM agent runs through Ollama on localhost.
# - Raw video frames are not sent to the cloud.
# - Only structured metadata is sent to the local LLM agent.
# ============================================================

from ultralytics import YOLO
import cv2
import time
import math
import requests
import json
import pandas as pd
import os


# =========================
# 1. Basic Configuration
# =========================

# Put your traffic video in the same folder as this notebook.
# Recommended file name: traffic.mp4
video_path = "traffic.mp4"

# If traffic.mp4 is not found, automatically use the first video file in the folder.
if not os.path.exists(video_path):
    video_files = [
        f for f in os.listdir()
        if f.lower().endswith((".mp4", ".mov", ".avi"))
    ]

    if len(video_files) == 0:
        raise FileNotFoundError("No video file found in the current folder.")

    video_path = video_files[0]
    print("traffic.mp4 not found. Using video:", video_path)

# Load YOLOv8n model
model = YOLO("yolov8n.pt")

# Local LLM model deployed through Ollama
LOCAL_LLM_MODEL = "qwen2.5:0.5b"

# COCO vehicle classes counted by the traffic control system
VEHICLE_CLASSES = {"car", "bus", "truck", "motorcycle"}

# Detection settings
SAMPLE_INTERVAL = 10       # sample one frame every 10 seconds
SAMPLES_PER_CYCLE = 3      # sample 3 frames per control cycle
MAX_ANALYSIS_TIME = 180    # analyze the first 180 seconds of video
MAX_CYCLES = 4             # limit cycles for presentation/demo
CONF_THRESHOLD = 0.20      # YOLO confidence threshold

# Traffic signal control settings
MIN_GREEN = 10             # minimum green light time in seconds
MAX_GREEN = 45             # maximum green light time in seconds
YELLOW_TIME = 3            # yellow light duration in seconds
EXTENSION_THRESHOLD = 5    # every 5 vehicles extends green time
EXTENSION_INCREMENT = 5    # green time extension step in seconds

# Output files
RESULT_CSV = "traffic_control_with_llm_results_fixed.csv"
DETECTION_IMAGE = "yolo_detection_sample.jpg"


# =========================
# 2. Vehicle Detection
# =========================

def count_vehicles(frame):
    """
    Count vehicles in one video frame using YOLOv8n.

    Only the following classes are counted:
    car, bus, truck, motorcycle.
    """

    start_time = time.time()

    results = model(
        frame,
        imgsz=1280,
        verbose=False,
        conf=CONF_THRESHOLD
    )

    latency_ms = (time.time() - start_time) * 1000

    vehicle_count = 0
    detected_objects = []

    for box in results[0].boxes:
        cls_id = int(box.cls[0])
        cls_name = model.names[cls_id]
        confidence = float(box.conf[0])

        detected_objects.append((cls_name, round(confidence, 2)))

        if cls_name in VEHICLE_CLASSES:
            vehicle_count += 1

    annotated_frame = results[0].plot()

    return vehicle_count, latency_ms, detected_objects, annotated_frame


# =========================
# 3. Adaptive Traffic Control
# =========================

def calculate_green_time(avg_vehicle_count):
    """
    Convert average vehicle count into adaptive green light duration.

    Rule:
    - If the average vehicle count is very low, use the minimum green time.
    - For every 5 vehicles above the threshold, extend green time by 5 seconds.
    - The green light time is limited between MIN_GREEN and MAX_GREEN.
    """

    if avg_vehicle_count <= EXTENSION_THRESHOLD:
        return MIN_GREEN

    extension_steps = math.ceil(
        (avg_vehicle_count - EXTENSION_THRESHOLD) / EXTENSION_THRESHOLD
    )

    green_time = MIN_GREEN + extension_steps * EXTENSION_INCREMENT

    return int(max(MIN_GREEN, min(MAX_GREEN, green_time)))


def get_traffic_state(avg_vehicle_count):
    """
    Determine traffic state using a deterministic rule.
    This avoids unstable LLM classification.
    """

    if avg_vehicle_count >= 35:
        return "Heavy traffic"
    elif avg_vehicle_count >= 20:
        return "Moderate traffic"
    elif avg_vehicle_count >= 5:
        return "Light traffic"
    else:
        return "Very low traffic"


def get_rule_based_decision(traffic_state, green_time):
    """
    Generate a stable rule-based decision and action.
    The LLM can explain the decision, but the core decision is controlled by code.
    """

    if traffic_state == "Heavy traffic":
        decision = "Extend green light to maximum or near-maximum duration."
        action = "Generate congestion warning and continue local monitoring."
    elif traffic_state == "Moderate traffic":
        decision = "Use extended green light duration."
        action = "Continue monitoring traffic flow."
    elif traffic_state == "Light traffic":
        decision = "Use normal or shorter green light duration."
        action = "Continue next detection cycle."
    else:
        decision = "Use minimum green light duration."
        action = "Switch quickly to the next signal phase."

    return decision, action


# =========================
# 4. Local LLM Agent
# =========================

def extract_json_from_text(text):
    """
    Extract JSON from local LLM output.

    This function helps when the LLM returns JSON inside markdown code blocks
    or adds extra text around the JSON object.
    """

    text = text.strip()

    if text.startswith("```"):
        text = text.replace("```json", "")
        text = text.replace("```", "")
        text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        json_text = text[start:end + 1]

        try:
            return json.loads(json_text)
        except json.JSONDecodeError:
            pass

    return None


def check_local_llm():
    """
    Check whether the local Ollama service is available.
    """

    try:
        response = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=10
        )

        if response.status_code == 200:
            print("Local LLM service is available.")
            print("Available Ollama models:")
            print(response.text)
            return True

        print("Local LLM service returned status code:", response.status_code)
        return False

    except Exception as e:
        print("Local LLM service is not available.")
        print("Reason:", e)
        print("The system will use local fallback decision output.")
        return False


def local_llm_traffic_agent(
    avg_count,
    green_time,
    avg_latency,
    sample_times,
    counts,
    traffic_state,
    rule_decision,
    rule_action
):
    """
    Local lightweight LLM agent for traffic decision explanation.

    Important design:
    - Python rule determines traffic_state.
    - Python rule determines core decision and action.
    - Local LLM explains the decision and provides a privacy note.
    - Raw video frames are not sent to the LLM.
    """

    prompt = f"""
You are a local Edge AI traffic decision agent.

The traffic state and traffic control decision have already been determined
by the local rule-based controller. Do not change them.

Structured traffic metadata:
- Sample times: {sample_times}
- Vehicle counts: {counts}
- Average vehicle count: {avg_count:.2f}
- Rule-based traffic state: {traffic_state}
- Green light time: {green_time} seconds
- Average inference latency: {avg_latency:.2f} ms
- Yellow light time: {YELLOW_TIME} seconds
- Rule-based decision: {rule_decision}
- Rule-based action: {rule_action}

Task:
Explain the decision based only on the structured metadata.
Output only valid JSON using this schema:

{{
  "traffic_state": "{traffic_state}",
  "decision": "{rule_decision}",
  "action": "{rule_action}",
  "reason": "...",
  "privacy_note": "..."
}}

Rules:
- Do not change the traffic_state.
- Do not change the decision.
- Do not change the action.
- The reason should explain why the green light time is reasonable.
- Mention that raw video is processed locally and is not sent to the cloud.
- Keep the answer concise.
"""

    url = "http://127.0.0.1:11434/api/chat"

    payload = {
        "model": LOCAL_LLM_MODEL,
        "messages": [
            {
                "role": "system",
                "content": "You are a concise local traffic control agent. Output only valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    try:
        llm_start = time.time()

        response = requests.post(
            url,
            json=payload,
            timeout=60
        )

        llm_latency_ms = (time.time() - llm_start) * 1000

        if response.status_code != 200:
            return {
                "traffic_state": traffic_state,
                "decision": rule_decision,
                "action": rule_action,
                "reason": f"Ollama returned status code {response.status_code}. Rule-based local decision was used.",
                "privacy_note": "Fallback decision was made locally without cloud processing.",
                "llm_latency_ms": round(llm_latency_ms, 2)
            }

        content = response.json()["message"]["content"].strip()
        parsed_json = extract_json_from_text(content)

        if parsed_json is None:
            parsed_json = {
                "traffic_state": traffic_state,
                "decision": rule_decision,
                "action": rule_action,
                "reason": content,
                "privacy_note": "Only structured metadata was sent to the local LLM agent. Raw video stayed local."
            }

        # Force stable rule-based values.
        # This prevents the small LLM from changing the traffic state incorrectly.
        parsed_json["traffic_state"] = traffic_state
        parsed_json["decision"] = rule_decision
        parsed_json["action"] = rule_action
        parsed_json["llm_latency_ms"] = round(llm_latency_ms, 2)

        if "reason" not in parsed_json or not parsed_json["reason"]:
            parsed_json["reason"] = (
                f"The average vehicle count is {avg_count:.2f}, "
                f"which indicates {traffic_state}. Therefore, a green light time "
                f"of {green_time} seconds is reasonable."
            )

        if "privacy_note" not in parsed_json or not parsed_json["privacy_note"]:
            parsed_json["privacy_note"] = (
                "Raw video is processed locally. Only structured metadata is used by the local LLM agent."
            )

        return parsed_json

    except Exception as e:
        return {
            "traffic_state": traffic_state,
            "decision": rule_decision,
            "action": rule_action,
            "reason": f"The local LLM agent could not be called: {str(e)}. Rule-based local decision was used.",
            "privacy_note": "Fallback decision was made locally without cloud processing.",
            "llm_latency_ms": None
        }


# =========================
# 5. Main Adaptive Loop
# =========================

local_llm_available = check_local_llm()

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise FileNotFoundError(f"Cannot open video file: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_duration = frame_count / fps if fps > 0 else 0

print("\n================ Video Information ================")
print("Video path:", video_path)
print("FPS:", round(fps, 2))
print("Frame count:", frame_count)
print("Video duration:", round(video_duration, 2), "seconds")
print("Start adaptive traffic control with local LLM agent...\n")

cycle_id = 1
current_time = 0
cycle_records = []
sample_image_saved = False

while current_time <= min(MAX_ANALYSIS_TIME, video_duration) and cycle_id <= MAX_CYCLES:
    counts = []
    latencies = []
    sample_times = []
    detected_objects_all = []

    print(f"========== Cycle {cycle_id} ==========")

    for i in range(SAMPLES_PER_CYCLE):
        sample_time = current_time + i * SAMPLE_INTERVAL

        if sample_time > min(MAX_ANALYSIS_TIME, video_duration):
            break

        cap.set(cv2.CAP_PROP_POS_FRAMES, int(sample_time * fps))
        ret, frame = cap.read()

        if not ret:
            print(f"Failed to read frame at {sample_time:.0f}s")
            break

        vehicle_count, latency_ms, detected_objects, annotated_frame = count_vehicles(frame)

        counts.append(vehicle_count)
        latencies.append(latency_ms)
        sample_times.append(round(sample_time, 2))
        detected_objects_all.append(detected_objects)

        print(f"Time {sample_time:.0f}s -> {vehicle_count} vehicles, latency {latency_ms:.2f} ms")

        if not sample_image_saved:
            cv2.imwrite(DETECTION_IMAGE, annotated_frame)
            sample_image_saved = True
            print(f"Sample YOLO detection image saved as {DETECTION_IMAGE}")

    if len(counts) == 0:
        print("No valid frames in this cycle. Stop analysis.")
        break

    avg_count = sum(counts) / len(counts)
    avg_latency = sum(latencies) / len(latencies)
    green_time = calculate_green_time(avg_count)

    traffic_state = get_traffic_state(avg_count)
    rule_decision, rule_action = get_rule_based_decision(traffic_state, green_time)

    print(f"Average vehicle count: {avg_count:.2f}")
    print(f"Average inference latency: {avg_latency:.2f} ms")
    print(f"Traffic state by local rule: {traffic_state}")
    print(f"Adaptive green light time: {green_time} seconds")
    print(f"Yellow light time: {YELLOW_TIME} seconds")
    print(f"Next detection starts after {green_time + YELLOW_TIME} seconds")
    print(f"Rule-based decision: {rule_decision}")
    print(f"Rule-based action: {rule_action}")

    agent_result = local_llm_traffic_agent(
        avg_count=avg_count,
        green_time=green_time,
        avg_latency=avg_latency,
        sample_times=sample_times,
        counts=counts,
        traffic_state=traffic_state,
        rule_decision=rule_decision,
        rule_action=rule_action
    )

    print("LLM Agent Traffic State:", agent_result.get("traffic_state"))
    print("LLM Agent Decision:", agent_result.get("decision"))
    print("LLM Agent Action:", agent_result.get("action"))
    print("LLM Agent Reason:", agent_result.get("reason"))
    print("Privacy Note:", agent_result.get("privacy_note"))
    print("LLM Agent Latency:", agent_result.get("llm_latency_ms"), "ms")
    print()

    cycle_records.append({
        "cycle": cycle_id,
        "sample_times": sample_times,
        "counts": counts,
        "average_count": round(avg_count, 2),
        "average_inference_latency_ms": round(avg_latency, 2),
        "traffic_state": traffic_state,
        "green_time_s": green_time,
        "yellow_time_s": YELLOW_TIME,
        "next_detection_after_s": green_time + YELLOW_TIME,
        "rule_decision": rule_decision,
        "rule_action": rule_action,
        "agent_traffic_state": agent_result.get("traffic_state"),
        "agent_decision": agent_result.get("decision"),
        "agent_action": agent_result.get("action"),
        "agent_reason": agent_result.get("reason"),
        "privacy_note": agent_result.get("privacy_note"),
        "llm_latency_ms": agent_result.get("llm_latency_ms"),
    })

    current_time += green_time + YELLOW_TIME
    cycle_id += 1

cap.release()


# =========================
# 6. Save Results
# =========================

df_results = pd.DataFrame(cycle_records)

print("================ Final Cycle Records ================")
print(df_results)

df_results.to_csv(RESULT_CSV, index=False)

print("\nResults saved as:", RESULT_CSV)
print("YOLO detection sample saved as:", DETECTION_IMAGE)
print("Adaptive traffic control with local LLM agent finished.")

Local LLM service is available.
Available Ollama models:
{"models":[{"name":"qwen2.5:0.5b","model":"qwen2.5:0.5b","modified_at":"2026-06-03T20:30:55.7015276+08:00","size":397821319,"digest":"a8b0c51577010a279d933d14c2a8ab4b268079d44c5c8830c0a93900f1827c67","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"494.03M","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":896},"capabilities":["completion","tools"]}]}

================ Video Information ================
Video path: traffic.mp4
FPS: 29.97
Frame count: 12433
Video duration: 414.85 seconds
Start adaptive traffic control with local LLM agent...

========== Cycle 1 ==========
Time 0s -> 111 vehicles, latency 511.02 ms
Sample YOLO detection image saved as yolo_detection_sample.jpg
Time 10s -> 100 vehicles, latency 170.53 ms
Time 20s -> 106 vehicles, latency 160.47 ms
Average vehicle count: 105.67
Average inference latency: 280.67 ms
Traffic state by local rule: H

In [1]:
import requests

try:
    response = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
    print("Status code:", response.status_code)
    print(response.text)
except Exception as e:
    print("Connection failed:")
    print(e)

Status code: 200
{"models":[{"name":"qwen2.5:0.5b","model":"qwen2.5:0.5b","modified_at":"2026-06-03T20:30:55.7015276+08:00","size":397821319,"digest":"a8b0c51577010a279d933d14c2a8ab4b268079d44c5c8830c0a93900f1827c67","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"494.03M","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":896},"capabilities":["completion","tools"]}]}


In [2]:
import requests
import json

url = "http://127.0.0.1:11434/api/chat"

payload = {
    "model": "qwen2.5:0.5b",
    "messages": [
        {
            "role": "user",
            "content": "Reply only with: local LLM agent is working."
        }
    ],
    "stream": False
}

print("Sending request to local Ollama...")

try:
    response = requests.post(url, json=payload, timeout=60)
    print("Status code:", response.status_code)

    data = response.json()
    print(data["message"]["content"])

except Exception as e:
    print("LLM request failed:")
    print(e)

Sending request to local Ollama...
Status code: 200
local LLM agent is working. How may I assist you today?
